In [ ]:
import sys
sys.path.append(".")

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

dtype_float = tf.float64
dtype_comp = tf.complex128
tf.keras.backend.set_floatx('float64')

import ED_utill as edut
import ED_tfi_1d as ed
import Energy_ham as enham
import model_sampler as samp
import utils_selector as sel

In [ ]:
# System parameters
L = 8
J = 1.0
g = 1.0
pbc = True

print(f'System: L={L}, J={J}, g={g}, pbc={pbc}')

In [ ]:
# Exact diagonalization ground state
E0_ed, psi0_ed = ed.calc_gs(L, g, J=J, pbc=pbc)
print(f'ED ground state energy: E0 = {E0_ed:.8f}')
print(f'ED ground state energy per site: E0/L = {E0_ed/L:.8f}')

In [ ]:
# Build NQS model (simple dense network)
def build_rbm(L, alpha=2):
    """Build a simple RBM-like dense model for log|psi(s)|."""
    inputs = tf.keras.Input(shape=(L,), dtype=tf.float64)
    x = tf.keras.layers.Dense(alpha * L, activation='tanh', dtype=tf.float64)(inputs)
    x = tf.keras.layers.Dense(1, dtype=tf.float64)(x)
    outputs = tf.squeeze(x, axis=-1)
    return tf.keras.Model(inputs, outputs)

model = build_rbm(L, alpha=2)
model.summary()

In [ ]:
# Ground state training
import gen_gs_train as gst

n_samples = 500
n_epochs = 500
lr = 0.01

energies = gst.train_gs(
    model,
    hamiltonian_name='local_energy_tf',
    sampler_name='metropolis_sample_tf',
    n_samples=n_samples,
    L=L,
    n_epochs=n_epochs,
    lr=lr,
    J=J,
    g=g,
    pbc=pbc
)

print(f'\nFinal VMC energy: {energies[-1]:.6f}')
print(f'ED  ground state: {E0_ed:.6f}')
print(f'Error: {abs(energies[-1] - E0_ed):.6f}')

In [ ]:
# SR optimization
import sr_gs_opt as sropt

model_sr = build_rbm(L, alpha=2)

energies_sr = sropt.sr_gs_optimize(
    model_sr,
    hamiltonian_name='local_energy_tf',
    sampler_name='metropolis_sample_tf',
    n_samples=n_samples,
    L=L,
    n_epochs=n_epochs,
    lr=lr,
    J=J,
    g=g,
    pbc=pbc
)

print(f'\nFinal SR energy: {energies_sr[-1]:.6f}')
print(f'ED  ground state: {E0_ed:.6f}')

In [ ]:
# Final operator evaluation
import op_final as opf

e_mean, e_err = opf.compute_operator_final(
    model_sr,
    operator_name='local_energy_tf',
    sampler_name='metropolis_sample_tf',
    n_samples=2000,
    L=L,
    J=J,
    g=g,
    pbc=pbc
)

print(f'Final energy estimate: E = {e_mean:.6f} +/- {e_err:.6f}')
print(f'ED ground state:       E = {E0_ed:.6f}')

In [ ]:
# Plot energy convergence
plt.figure(figsize=(10, 5))
plt.plot(energies, label='VMC (SGD)')
plt.plot(energies_sr, label='VMC (SR)')
plt.axhline(E0_ed, color='k', linestyle='--', label='ED (exact)')
plt.xlabel('Epoch')
plt.ylabel('Energy')
plt.title(f'TFIM Ground State Training  L={L}, g={g}')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Sampling diagnostics
samples = samp.metropolis_sample_tf(model_sr, 1000, L, n_therm=200)
print(f'Sampled {samples.shape[0]} configurations')
print(f'Mean magnetization: {tf.reduce_mean(samples).numpy():.4f}')
print(f'Mean |m|: {tf.reduce_mean(tf.abs(tf.reduce_mean(samples, axis=1))).numpy():.4f}')